# Create / Seed `recommendation_catalog`

**Default lakehouse:** ManagedServiceData

This notebook creates and (re)seeds the `recommendation_catalog` Delta table used by the **AssessmentSalesAdvisor** Fabric data agent.

## Purpose
Every recommendation the agent makes must be backed by a row in this table. The table maps assessment findings (failed/warning checks or low-scoring categories) to:
- A Microsoft SKU or service to upsell / cross-sell
- A short sales talking point the AE can use
- An `impact_weight` (`Very High` / `High` / `Medium` / `Low`) used to rank opportunities

## Behaviour
- **Idempotent:** uses `MERGE` on `rule_id`; safe to re-run.
- **Editable:** existing rules are updated, new rules inserted, nothing deleted automatically.
- **Soft-deletable:** flip `active = false` instead of deleting rules you no longer want active.

## Matching semantics (consumed by the agent)
A rule matches a check row when **all** of these are true:
- `active = true`
- `assessment_type` is `'*'` OR equals the assessment's `assessment_type`
- `match_category` is `'*'` OR **any `|`-separated token** appears (case-insensitive substring) in the check's `category`
- `match_keyword` is `'*'` OR **any `|`-separated token** appears (case-insensitive substring) in `check_name` + `tags` + `framework_control`
- check `status` is `'Failed'` or `'Warning'`

Use `|` inside `match_category` / `match_keyword` to express alternatives, e.g. `Defender|Threat` or `Safe Links|Phishing`.

**Special case:** the `Managed Security Service` rule does not match per-check — it triggers when the tenant's latest `overall_score_pct < 60`. The agent applies that threshold explicitly.

> **Note:** No prices or ARR figures are stored. The catalog is purely product-mapping + narrative + impact ranking.

In [8]:
# --- Create the Delta table if it doesn't exist ------------------------------
CATALOG_TABLE = "recommendation_catalog"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG_TABLE} (
        rule_id              STRING  NOT NULL,
        assessment_type      STRING  NOT NULL,   -- 'Copilot Readiness' | 'Copilot Assessment' | 'Security Assessment' | '*'
        match_category       STRING  NOT NULL,   -- substring matched against check.category (case-insensitive); '*' = any
        match_keyword        STRING  NOT NULL,   -- substring matched against check_name + tags + framework_control; '*' = any
        recommendation_sku   STRING  NOT NULL,   -- product / SKU / service name to pitch
        recommendation_type  STRING  NOT NULL,   -- 'License Upsell' | 'Add-on' | 'Managed Service' | 'Workshop' | 'Prereq Upsell'
        talking_point        STRING  NOT NULL,   -- one-sentence sales narrative
        impact_weight        STRING  NOT NULL,   -- 'Very High' | 'High' | 'Medium' | 'Low'
        active               BOOLEAN NOT NULL,
        created_at           TIMESTAMP,
        updated_at           TIMESTAMP
    ) USING DELTA
""")
print(f"✓ Table ensured: {CATALOG_TABLE}")


StatementMeta(, 20161698-afbc-4f50-9ba9-b2cf5e9dd54b, 7, Finished, Available, Finished, False)

✓ Table ensured: recommendation_catalog


In [ ]:
# --- Seed rules --------------------------------------------------------------
# Add / edit rows here. rule_id MUST be unique and stable; reusing an id will
# update that rule on the next run. Set active=False to retire a rule without
# deleting history.
#
# Column order:
#   rule_id, assessment_type, match_category, match_keyword,
#   recommendation_sku, recommendation_type, talking_point, impact_weight

from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, TimestampType
)

SEED_RULES = [
    # Catalog is denormalized: one token per match_category / match_keyword
    # so the agent's join is a plain LIKE (no STRING_SPLIT / CROSS APPLY).

    # ----- Security Assessment ---------------------------------------------
    ("sec-identity-mfa",
     "Security Assessment", "Identity", "MFA",
     "Entra ID P1", "License Upsell",
     "Enforce MFA org-wide; included in Entra ID P1 / M365 E3 step-up.",
     "High"),
    ("sec-identity-ca",
     "Security Assessment", "Identity", "Conditional Access",
     "Entra ID P1", "License Upsell",
     "Conditional Access requires Entra ID P1.",
     "High"),

    # PIM: Identity x {Privileged, PIM}
    ("sec-identity-pim-1",
     "Security Assessment", "Identity", "Privileged",
     "Entra ID P2", "Add-on",
     "Privileged Identity Management is P2-only.",
     "Medium"),
    ("sec-identity-pim-2",
     "Security Assessment", "Identity", "PIM",
     "Entra ID P2", "Add-on",
     "Privileged Identity Management is P2-only.",
     "Medium"),

    # Defender Email: {Defender, Threat} x {Safe Links, Phishing}
    ("sec-defender-email-1",
     "Security Assessment", "Defender", "Safe Links",
     "Defender for Office 365 P2", "Add-on",
     "Safe Links / Safe Attachments + automated investigation.",
     "High"),
    ("sec-defender-email-2",
     "Security Assessment", "Defender", "Phishing",
     "Defender for Office 365 P2", "Add-on",
     "Safe Links / Safe Attachments + automated investigation.",
     "High"),
    ("sec-defender-email-3",
     "Security Assessment", "Threat", "Safe Links",
     "Defender for Office 365 P2", "Add-on",
     "Safe Links / Safe Attachments + automated investigation.",
     "High"),
    ("sec-defender-email-4",
     "Security Assessment", "Threat", "Phishing",
     "Defender for Office 365 P2", "Add-on",
     "Safe Links / Safe Attachments + automated investigation.",
     "High"),

    # Endpoint EDR: Endpoint x {EDR, Endpoint Detection}
    ("sec-endpoint-edr-1",
     "Security Assessment", "Endpoint", "EDR",
     "Defender for Endpoint P2", "Add-on",
     "EDR with auto-remediation; part of M365 E5 Security.",
     "High"),
    ("sec-endpoint-edr-2",
     "Security Assessment", "Endpoint", "Endpoint Detection",
     "Defender for Endpoint P2", "Add-on",
     "EDR with auto-remediation; part of M365 E5 Security.",
     "High"),

    # Data DLP: {Data Protection, DLP} x {Sensitivity, DLP}
    ("sec-data-dlp-1",
     "Security Assessment", "Data Protection", "Sensitivity",
     "Purview Information Protection", "Add-on",
     "Sensitivity labels + DLP across M365 and endpoints.",
     "Very High"),
    ("sec-data-dlp-2",
     "Security Assessment", "Data Protection", "DLP",
     "Purview Information Protection", "Add-on",
     "Sensitivity labels + DLP across M365 and endpoints.",
     "Very High"),
    ("sec-data-dlp-3",
     "Security Assessment", "DLP", "Sensitivity",
     "Purview Information Protection", "Add-on",
     "Sensitivity labels + DLP across M365 and endpoints.",
     "Very High"),
    ("sec-data-dlp-4",
     "Security Assessment", "DLP", "DLP",
     "Purview Information Protection", "Add-on",
     "Sensitivity labels + DLP across M365 and endpoints.",
     "Very High"),

    # Insider Risk: {Governance, Insider} x Insider Risk
    ("sec-insider-risk-1",
     "Security Assessment", "Governance", "Insider Risk",
     "Priva + Insider Risk", "Add-on",
     "Privacy posture + insider risk in M365 E5 Compliance.",
     "Medium"),
    ("sec-insider-risk-2",
     "Security Assessment", "Insider", "Insider Risk",
     "Priva + Insider Risk", "Add-on",
     "Privacy posture + insider risk in M365 E5 Compliance.",
     "Medium"),

    # ----- Copilot Readiness ------------------------------------------------
    ("cr-identity-baseline",
     "Copilot Readiness", "Identity", "Identity Baseline",
     "Entra ID P1+", "Prereq Upsell",
     "Copilot needs strong identity baseline first.",
     "Very High"),

    # CR Data Labels: Data x {Sensitivity, Label}
    ("cr-data-labels-1",
     "Copilot Readiness", "Data", "Sensitivity",
     "Purview Information Protection", "Prereq Upsell",
     "Without labels, Copilot oversharing risk is high.",
     "Very High"),
    ("cr-data-labels-2",
     "Copilot Readiness", "Data", "Label",
     "Purview Information Protection", "Prereq Upsell",
     "Without labels, Copilot oversharing risk is high.",
     "Very High"),

    # CR Adoption: Adoption x {Adoption, Enablement}
    ("cr-adoption-1",
     "Copilot Readiness", "Adoption", "Adoption",
     "Copilot Adoption Workshop", "Workshop",
     "Run a 2-week adoption pilot to drive measurable usage.",
     "High"),
    ("cr-adoption-2",
     "Copilot Readiness", "Adoption", "Enablement",
     "Copilot Adoption Workshop", "Workshop",
     "Run a 2-week adoption pilot to drive measurable usage.",
     "High"),

    # ----- Intune / Endpoint Management ------------------------------------
    # Device-management findings → Intune (M365 E3 → E5 step-up).
    ("sec-intune-device-1",
     "Security Assessment", "Device", "Intune",
     "Microsoft Intune", "License Upsell",
     "Centralized device compliance & app protection via Intune; foundational E3→E5 step-up.",
     "High"),
    ("sec-intune-device-2",
     "Security Assessment", "Device", "MDM",
     "Microsoft Intune", "License Upsell",
     "Centralized device compliance & app protection via Intune; foundational E3→E5 step-up.",
     "High"),
    ("sec-intune-device-3",
     "Security Assessment", "Device", "Compliance Policy",
     "Microsoft Intune", "License Upsell",
     "Centralized device compliance & app protection via Intune; foundational E3→E5 step-up.",
     "High"),
    ("sec-intune-endpoint-1",
     "Security Assessment", "Endpoint", "Intune",
     "Microsoft Intune", "License Upsell",
     "Centralized device compliance & app protection via Intune; foundational E3→E5 step-up.",
     "High"),
    ("sec-intune-endpoint-2",
     "Security Assessment", "Endpoint", "Device Compliance",
     "Microsoft Intune", "License Upsell",
     "Centralized device compliance & app protection via Intune; foundational E3→E5 step-up.",
     "High"),
    ("sec-intune-mobile-1",
     "Security Assessment", "Mobile", "Intune",
     "Microsoft Intune", "License Upsell",
     "Centralized device compliance & app protection via Intune; foundational E3→E5 step-up.",
     "High"),
    ("sec-intune-mobile-2",
     "Security Assessment", "Mobile", "App Protection",
     "Microsoft Intune", "License Upsell",
     "Centralized device compliance & app protection via Intune; foundational E3→E5 step-up.",
     "High"),

    # ----- Defender for Cloud Apps (CASB) ----------------------------------
    ("sec-mdca-cloudapps-1",
     "Security Assessment", "Cloud Apps", "OAuth",
     "Defender for Cloud Apps", "Add-on",
     "CASB controls for OAuth apps and Shadow IT; visibility across SaaS.",
     "High"),
    ("sec-mdca-cloudapps-2",
     "Security Assessment", "Cloud Apps", "Shadow IT",
     "Defender for Cloud Apps", "Add-on",
     "CASB controls for OAuth apps and Shadow IT; visibility across SaaS.",
     "High"),
    ("sec-mdca-cloudapps-3",
     "Security Assessment", "Cloud Apps", "Sanctioned",
     "Defender for Cloud Apps", "Add-on",
     "CASB controls for OAuth apps and Shadow IT; visibility across SaaS.",
     "High"),
    ("sec-mdca-saas-1",
     "Security Assessment", "SaaS", "OAuth",
     "Defender for Cloud Apps", "Add-on",
     "CASB controls for OAuth apps and Shadow IT; visibility across SaaS.",
     "High"),
    ("sec-mdca-saas-2",
     "Security Assessment", "SaaS", "Shadow IT",
     "Defender for Cloud Apps", "Add-on",
     "CASB controls for OAuth apps and Shadow IT; visibility across SaaS.",
     "High"),
    ("sec-mdca-shadow",
     "Security Assessment", "Shadow IT", "Discovery",
     "Defender for Cloud Apps", "Add-on",
     "CASB controls for OAuth apps and Shadow IT; visibility across SaaS.",
     "High"),

    # ----- Compliance Manager / Purview Audit Premium ----------------------
    ("sec-audit-1",
     "Security Assessment", "Audit", "Audit Log",
     "Purview Audit Premium", "Add-on",
     "Long-retention audit logs + advanced audit events for forensic investigations.",
     "Medium"),
    ("sec-audit-2",
     "Security Assessment", "Audit", "Retention",
     "Purview Audit Premium", "Add-on",
     "Long-retention audit logs + advanced audit events for forensic investigations.",
     "Medium"),
    ("sec-audit-3",
     "Security Assessment", "Audit", "Mailbox Audit",
     "Purview Audit Premium", "Add-on",
     "Long-retention audit logs + advanced audit events for forensic investigations.",
     "Medium"),
    ("sec-compliance-1",
     "Security Assessment", "Compliance", "Compliance Score",
     "Compliance Manager", "Add-on",
     "Compliance Manager scores posture against ISO/NIST/CIS and tracks improvement actions.",
     "Medium"),
    ("sec-compliance-2",
     "Security Assessment", "Compliance", "Retention",
     "Compliance Manager", "Add-on",
     "Compliance Manager scores posture against ISO/NIST/CIS and tracks improvement actions.",
     "Medium"),
    ("sec-governance-audit",
     "Security Assessment", "Governance", "Audit",
     "Purview Audit Premium", "Add-on",
     "Long-retention audit logs + advanced audit events for forensic investigations.",
     "Medium"),

    # ----- Cross-cutting posture --------------------------------------------
    # Special-case rule: triggers when overall_score_pct < 60.
    ("any-low-score-mss",
     "*", "Security Posture", "Overall score < 60",
     "Managed Security Service", "Managed Service",
     "Score is below industry baseline — recommend ongoing managed service.",
     "High"),

    # Special-case rule: triggers when a tenant has very high failed-check
    # volume (e.g. failed_count >= 50). The agent applies the threshold; the
    # rule is here so the SKU + talking point are catalog-grounded.
    ("any-high-failures-sentinel",
     "*", "Security Operations", "High failure volume",
     "Microsoft Sentinel", "Managed Service",
     "High failure volume warrants a SIEM/SOAR — Sentinel for centralized detection & response.",
     "High"),
]

now = datetime.now(timezone.utc)
seed_rows = [
    Row(
        rule_id=r[0],
        assessment_type=r[1],
        match_category=r[2],
        match_keyword=r[3],
        recommendation_sku=r[4],
        recommendation_type=r[5],
        talking_point=r[6],
        impact_weight=r[7],
        active=True,
        created_at=now,
        updated_at=now,
    )
    for r in SEED_RULES
]

seed_schema = StructType([
    StructField("rule_id",             StringType(),    False),
    StructField("assessment_type",     StringType(),    False),
    StructField("match_category",      StringType(),    False),
    StructField("match_keyword",       StringType(),    False),
    StructField("recommendation_sku",  StringType(),    False),
    StructField("recommendation_type", StringType(),    False),
    StructField("talking_point",       StringType(),    False),
    StructField("impact_weight",       StringType(),    False),
    StructField("active",              BooleanType(),   False),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
])

seed_df = spark.createDataFrame(seed_rows, schema=seed_schema)
seed_df.createOrReplaceTempView("recommendation_catalog_seed")

spark.sql(f"""
    MERGE INTO {CATALOG_TABLE} t
    USING recommendation_catalog_seed s
    ON t.rule_id = s.rule_id
    WHEN MATCHED THEN UPDATE SET
        t.assessment_type     = s.assessment_type,
        t.match_category      = s.match_category,
        t.match_keyword       = s.match_keyword,
        t.recommendation_sku  = s.recommendation_sku,
        t.recommendation_type = s.recommendation_type,
        t.talking_point       = s.talking_point,
        t.impact_weight       = s.impact_weight,
        t.active              = s.active,
        t.updated_at          = s.updated_at
    WHEN NOT MATCHED THEN INSERT *
""")
print(f"✓ Seeded {len(seed_rows)} rules into {CATALOG_TABLE}")

# Deactivate any rule_id that is no longer in the seed list (e.g. retired
# compound rules replaced by denormalized single-token rules). Delta UPDATE
# can't take a subquery, so use MERGE with WHEN NOT MATCHED BY SOURCE.
spark.sql(f"""
    MERGE INTO {CATALOG_TABLE} t
    USING recommendation_catalog_seed s
    ON t.rule_id = s.rule_id
    WHEN NOT MATCHED BY SOURCE AND t.active = true THEN
        UPDATE SET active = false, updated_at = current_timestamp()
""")
deactivated = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {CATALOG_TABLE} t
    WHERE t.active = false
      AND t.rule_id NOT IN (SELECT rule_id FROM recommendation_catalog_seed)
""").collect()[0]["n"]
print(f"✓ Total stale (now inactive) rules: {deactivated}")

StatementMeta(, 20161698-afbc-4f50-9ba9-b2cf5e9dd54b, 8, Finished, Available, Finished, False)

✓ Seeded 42 rules into recommendation_catalog
✓ Total stale (now inactive) rules: 7


In [3]:
# --- Inspect the catalog -----------------------------------------------------
print("Catalog summary:")
spark.sql(f"""
    SELECT recommendation_type, COUNT(*) AS rules
    FROM {CATALOG_TABLE}
    WHERE active = true
    GROUP BY recommendation_type
    ORDER BY rules DESC
""").show(truncate=False)

print("All active rules:")
display(spark.sql(f"""
    SELECT rule_id, assessment_type, match_category, match_keyword,
           recommendation_sku, recommendation_type, impact_weight, talking_point
    FROM {CATALOG_TABLE}
    WHERE active = true
    ORDER BY
        CASE impact_weight WHEN 'Very High' THEN 1 WHEN 'High' THEN 2
                          WHEN 'Medium' THEN 3 WHEN 'Low' THEN 4 ELSE 5 END,
        assessment_type, recommendation_sku
"""))

StatementMeta(, c49fd631-3f9d-4bbe-900c-dcb8eaac4101, 9, Finished, Available, Finished, False)

Catalog summary:
+-------------------+-----+
|recommendation_type|rules|
+-------------------+-----+
|Add-on             |5    |
|Prereq Upsell      |2    |
|License Upsell     |2    |
|Workshop           |1    |
|Managed Service    |1    |
+-------------------+-----+

All active rules:


SynapseWidget(Synapse.DataFrame, a7469042-f8ef-4151-9b55-314e38ea6101)

In [ ]:
# --- Smoke test: run the agent's join against a sample tenant ----------------
# Replace 'Acme Corp' with a real tenant name from your data to validate the
# matching logic before publishing the Fabric data agent.
#
# Matching semantics:
#   - match_category / match_keyword may contain '|'-delimited alternatives
#     (e.g. 'Defender|Threat'). A rule matches if ANY token is a case-insensitive
#     substring of the corresponding check field.
#   - '*' means match anything.
#   - The cross-cutting low-score rule (assessment_type='*', match_keyword='Overall score < 60')
#     is handled separately by checking overall_score_pct.

SAMPLE_TENANT = "Acme Corp"

spark.sql(f"""
WITH latest_sec AS (
  SELECT * FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY tenant_name ORDER BY assessment_date DESC) AS rn
    FROM security_assessment_assessments
    WHERE LOWER(tenant_name) = LOWER('{SAMPLE_TENANT}')
  ) WHERE rn = 1
),
findings AS (
  SELECT c.tenant_name, c.category, c.check_name, c.priority, c.tags,
         c.framework_control, c.status, a.overall_score_pct
  FROM security_assessment_checks c
  JOIN latest_sec a ON c.assessment_id = a.assessment_id
  WHERE c.status IN ('Failed','Warning')
),
matched_rules AS (
  SELECT
    r.recommendation_sku,
    r.recommendation_type,
    r.talking_point,
    r.impact_weight,
    f.check_name
  FROM findings f
  JOIN {CATALOG_TABLE} r
    ON r.active = true
   AND r.recommendation_sku <> 'Managed Security Service'  -- handled separately below
   AND (r.assessment_type IN ('Security Assessment','*'))
   AND (
        r.match_category = '*'
        OR EXISTS (
            SELECT 1
            FROM (SELECT EXPLODE(SPLIT(LOWER(r.match_category), '\\\\|')) AS tok) t
            WHERE LOWER(f.category) LIKE CONCAT('%', t.tok, '%')
        )
   )
   AND (
        r.match_keyword = '*'
        OR EXISTS (
            SELECT 1
            FROM (SELECT EXPLODE(SPLIT(LOWER(r.match_keyword), '\\\\|')) AS tok) t
            WHERE LOWER(CONCAT_WS(' ',
                                  f.check_name,
                                  COALESCE(f.tags,''),
                                  COALESCE(f.framework_control,'')))
                  LIKE CONCAT('%', t.tok, '%')
        )
   )
),
low_score_rule AS (
  SELECT r.recommendation_sku, r.recommendation_type, r.talking_point, r.impact_weight,
         CONCAT('Overall score = ', CAST(MAX(f.overall_score_pct) AS STRING), '%') AS check_name
  FROM findings f
  CROSS JOIN {CATALOG_TABLE} r
  WHERE r.active = true
    AND r.recommendation_sku = 'Managed Security Service'
    AND f.overall_score_pct < 60
  GROUP BY r.recommendation_sku, r.recommendation_type, r.talking_point, r.impact_weight
),
all_matched AS (
  SELECT * FROM matched_rules
  UNION ALL
  SELECT * FROM low_score_rule
)
SELECT recommendation_sku, recommendation_type, impact_weight,
       COUNT(*) AS supporting_findings,
       SLICE(COLLECT_LIST(check_name), 1, 3) AS top_evidence,
       FIRST(talking_point) AS talking_point
FROM all_matched
GROUP BY recommendation_sku, recommendation_type, impact_weight, talking_point
ORDER BY
    CASE impact_weight WHEN 'Very High' THEN 1 WHEN 'High' THEN 2
                      WHEN 'Medium' THEN 3 WHEN 'Low' THEN 4 ELSE 5 END,
    supporting_findings DESC
""").show(truncate=False)